## Testing the pipeline
This notebook aims to test how reliable the knime2galaxy pipeline yields results.

### 1. Test the same knime sample multiple times to check how often the translation is successfull

In [ ]:
from imaging_knime_to_galaxy.translate import translate_knime_to_galaxy
from pathlib import Path

# set path to the data/ folder 
data_folder = Path("../data/train_data_workflows/").resolve()

In [ ]:
# run the knime2galaxy pipeline i=5 times
for i in range(5):
    workflow = translate_knime_to_galaxy(
        knwf_path=data_folder / "file_to_translate" / "2025_03_2D_spot_detection.knwf",
        tools_metadata_path=data_folder / "tools_metadata.json",
        translation_table_path=data_folder / "translation_table.yml",
        workflow_examples_yml_path=data_folder / "workflow_translation_table.yml",
        output_galaxy_workflow_path=data_folder / "output_file" / f"test_i={i}_knime2galaxy_output.ga",
    )

The generated .ga files can not be checked manually or loaded into the Galaxy instance to see if it is possible or if any Errors are occurring.

### 2. Test several samples to check how the pipeline handles other examples
For this, the workflows from the __additional_workflows__ branch are used and integrated into the current branch. They are stored under [Galaxy](simple_workflow/Galaxy.tar.xz) and [Knime](simple_workflow/KNIME.tar.xz) respectively.

In [ ]:
# set paths to galaxy and knime workflow files
galaxy_path = data_folder / "Galaxy_2"
knime_path = data_folder / "Knime_2"

In [ ]:
import re
import os

# list all all available .knwf files in the KNIME path
files = os.listdir(knime_path)
regex_names = []

# filter out the filename without the file extension
for f in files:
    name = re.sub(r'^\d{4}_\d{2}_|\.knwf$', '', f)
    regex_names.append(name)
    print(name)

In [ ]:
# list all files in the output directory
finished_files = os.listdir(data_folder / "output_file" / "planemo_test/")

# go through all available .knwf files in the knime path
for i,file in enumerate(files):
    filename = name = re.sub(r'^\d{4}_\d{2}_|\.knwf$', '', file)
    filename_ga = filename + "_v2.ga" # get corresponding .ga file name

    # check whether the current file was already translated to .ga
    if filename_ga in finished_files:
        print(f"File {file} already processed - skipping")
        continue 

    # if this is not the case, send the .knwf file to the translation pipeline
    else:
        workflow = translate_knime_to_galaxy(
            knwf_path= knime_path / f"{file}",
            tools_metadata_path=data_folder / "tools_metadata.json",
            translation_table_path=data_folder / "translation_table.yml",
            workflow_examples_yml_path=data_folder / "workflow_translation_table.yml",
            output_galaxy_workflow_path=data_folder / "output_file" / "planemo_test" / f"{regex_names[i]}_v2.ga",
        )

In [ ]:
# use worklflow_lint to check for valid .ga file structure

import subprocess

results = []
outputs = os.listdir(data_folder / "output_file" / "planemo_test/")
valid_results = 0

for output in outputs:
    # ignore other than the desired .ga files
    if output.startswith("."):
        continue

    # run planemo's workflow_lint function and use subprocess, as planemo is a command line tool
    result = subprocess.run(
        ["planemo", "workflow_lint", data_folder / "output_file" / "planemo_test" / f"{output}"],
        capture_output=True,
        text=True
    )
    stdout = result.stdout
    stderr = result.stderr
    has_error = "ERROR" in stdout or "ERROR" in stderr # check whether the result holds an error

    if has_error:
        print("Lint errors present for workflow", output)
        print(stdout)
    else:
        print("Lint passed for workflow", output)
        valid_results += 1

    results.append(result)
print(f"Valid results: {valid_results} out of {len(files)*2}")

In [ ]:
# check whether the error tools are part of the prompting data or whether it is hallucinated by the model

import json

# name of the tools that raise an error
error_tool_1 = "toolshed.g2.bx.psu.edu/repos/imgteam/img_writer/ip_image_writer/0.1+galaxy0"
error_tool_2 = "toolshed.g2.bx.psu.edu/repos/imgteam/median_filter/ip_median_filter/0.1+galaxy0"
error_tool_3 = "toolshed.g2.bx.psu.edu/repos/imgteam/binary_morphology/ip_binary_morphology/0.1+galaxy0"

# plus one valid tool fetched from the toolshed -> serves as proof of concept 
correct_tool = "toolshed.g2.bx.psu.edu/repos/imgteam/bfconvert/ip_convertimage/6.7.0+galaxy3" 

# open the tool metadata file
with open(data_folder / "tools_metadata.json") as f:
    data = json.load(f)

# function to search for the given tools in the tool metadata file (if it is found there, the tools are not hallucinated by the model)
def search_values(obj, search_term):
    if isinstance(obj, dict):
        for v in obj.values():
            if search_values(v, search_term):
                return True
    elif isinstance(obj, list):
        for item in obj:
            if search_values(item, search_term):
                return True
    elif isinstance(obj, str):
        if search_term in obj:
            return True
    return False

# if False, the tools were hallucinated by the model and not given in the prompt
print(search_values(data, error_tool_1)) 
print(search_values(data, error_tool_2))
print(search_values(data, error_tool_3))
print(search_values(data, correct_tool))